# Installing and Importing Libraries


In [0]:
%pip install databricks-feature-engineering


In [0]:
dbutils.library.restartPython()


In [0]:
from databricks.feature_engineering import FeatureEngineeringClient


# Ingestion Functions


This step automates **creation and update of Feature Store tables** for different reference dates.

The function receives the table, SQL file, and a list of dates. For each date, the query is executed and the result is inserted into the Feature Store.

If the table does not yet exist, it is **created with primary keys and partitioned by DATA_REF**. If it already exists, data for new dates is added through merge.

This way, the function allows **reprocessing and updating the Feature Store in a standardized way**, avoiding manual execution of queries for each period.


In [0]:
# List of reference dates for Feature Store update
dates = ['2018-10', '2018-11', '2018-12', '2019-01', '2019-02', '2019-03', '2019-04', '2019-05', '2019-06', '2019-07', '2019-08', '2019-09', '2019-10', '2019-11', '2019-12', '2020-01', '2020-02', '2020-03', '2020-04', '2020-05', '2020-06', '2020-07', '2020-08', '2020-09', '2020-10', '2020-11', '2020-12', '2021-01', '2021-02', '2021-03', '2021-04', '2021-05', '2021-06']


In [0]:
def update_feature_store(catalog, database, table, query_file, reference_dates):
    table_name = f"{catalog}.{database}.{table}"

    # Read the SQL file containing the parameterized query
    with open(query_file, "r") as file:
        sql_query = file.read()

    fe_client = FeatureEngineeringClient()

    # Check if the table already exists in the Feature Store
    def does_table_exist(catalog, database, table):
        return spark.sql(
            f"SHOW TABLES FROM {catalog}.{database} LIKE '{table}'"
        ).count() > 0

    # Dataset primary and partition column names in English
    dataset_primary_keys = ["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"]
    dataset_partition_columns = ["REF_DATE"]

    if does_table_exist(catalog, database, table):
        # Update the existing table for each reference date
        for ref_date in reference_dates:
            print(f"Writing {ref_date}")
            result_df = spark.sql(
                sql_query.format(dt_ref=f"{ref_date}-01")
            )
            fe_client.write_table(
                name=table_name,
                df=result_df,
                mode="merge"
            )
        print("Table updated")
    else:
        # Create the table in the Feature Store with the first date
        first_ref_date = reference_dates[0]
        result_df = spark.sql(
            sql_query.format(dt_ref=f"{first_ref_date}-01")
        )
        fe_client.create_table(
            name=table_name,
            primary_keys=dataset_primary_keys,
            partition_columns=dataset_partition_columns,
            df=result_df
        )
        print("Table created")
        # Write the remaining dates to the newly created table
        for ref_date in reference_dates[1:]:
            print(f"Writing {ref_date}")
            result_df = spark.sql(
                sql_query.format(dt_ref=f"{ref_date}-01")
            )
            fe_client.write_table(
                name=table_name,
                df=result_df,
                mode="merge"
            )
        print("Table created")


# Data Ingestion


In this step, the ingestion function is used to **create and update the different Feature Store tables**.

Each feature set has its own SQL query and table, and all are processed for every reference date defined earlier.

The following sources are updated:

| Feature Store             | Content                                |
| ------------------------- | -------------------------------------- |
| fs_cadastral              | Registration information               |
| fs_temporal               | Temporal information                   |
| fs_historico_financeiro   | Financial history                      |
| fs_renda                  | Income information                     |
| fs_funcionarios           | Employee-related information           |
| fs_historico_pagamentos   | Payment history                        |

This way, the different sources are **processed and stored in an organized way in the Feature Store**, becoming available for model training and prediction steps.


## Cadastral


In [0]:
update_feature_store(
    catalog="feature_store",
    database="credit_score",
    table="fs_cadastral",
    query_file="fs_cadastral.sql",
    dates=dates
)


## Temporal


In [0]:
update_feature_store(
    catalog="feature_store",
    database="credit_score",
    table="fs_temporal",
    query_file="fs_temporal.sql",
    dates=dates
)


## Income History


In [0]:
update_feature_store(
    catalog="feature_store",
    database="credit_score",
    table="fs_income_history",
    query_file="fs_income_history.sql",
    dates=dates
)


## Income


In [0]:
update_feature_store(
    catalog="feature_store",
    database="credit_score",
    table="fs_income",
    query_file="fs_income.sql",
    dates=dates
)


## Employees


In [0]:
update_feature_store(
    catalog="feature_store",
    database="credit_score",
    table="fs_employees",
    query_file="fs_employees.sql",
    dates=dates
)


## Payment History


In [0]:
update_feature_store(
    catalog="feature_store",
    database="credit_score",
    table="fs_payment_history",
    query_file="fs_payment_history.sql",
    dates=dates
)


# Final Ingestion Before Prediction


This step performs **final data ingestion into the Feature Store after model training is complete**.

Execution should occur only after training, ensuring that the features used in prediction are **updated with the most recent data available**.

This way, the flow is split into two steps: first **model training and evaluation** is performed, and only after it is complete is the **final Feature Store update** done, which will be used to generate predictions.


In [0]:
# Reference date for final feature ingestion for prediction
dates = ['2021-07']

# List of tables and corresponding SQL files (names in English)
tables_queries = [
    ("fs_cadastral", "fs_cadastral.sql"),
    ("fs_temporal", "fs_temporal.sql"),
    ("fs_income_history", "fs_income_history.sql"),
    ("fs_income", "fs_income.sql"),
    ("fs_employees", "fs_employees.sql"),
    ("fs_payment_history", "fs_payment_history.sql")
]

# Ingest final features into the Feature Store for prediction
for table, query_file in tables_queries:
    update_feature_store(
        catalog="feature_store",
        database="credit_score",
        table=table,
        query_file=query_file,
        dates=dates
    )
